In [1]:
import os
import pandas as pd
from ytmusicapi import YTMusic


In [2]:
yt = YTMusic('../headers_auth.json')
search_results = pd.DataFrame(yt.search("Shine on you crazy diamond"))
search_results.head(3)


,resultType,videoId,title,artists,album,duration,thumbnails,browseId,type,artist,year,author,itemCount,views
0,song,54W8kktFE_o,Shine On You Crazy Diamond (Parts I-V),"[{'name': 'Pink Floyd', 'id': 'UCO6LS_5W7vqG9m...","{'name': 'Wish You Were Here', 'id': 'MPREb_w7...",13:33,[{'url': 'https://lh3.googleusercontent.com/-x...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,song,9pkh8zi_eic,Shine On You Crazy Diamond (Parts 1 - 7) [Edit...,"[{'name': 'Pink Floyd', 'id': 'UCO6LS_5W7vqG9m...","{'name': 'Echoes (The Best Of Pink Floyd)', 'i...",17:32,[{'url': 'https://lh3.googleusercontent.com/B4...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,song,nHER1RmSqA8,Shine On You Crazy Diamond,"[{'name': 'Pink Floyd', 'id': 'UCO6LS_5W7vqG9m...","{'name': 'Pulse (Live)', 'id': 'MPREb_dCMssyAc...",13:35,[{'url': 'https://lh3.googleusercontent.com/qa...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
%%time

PLAYLIST_LIMIT=500
playlists = pd.DataFrame(yt.get_library_playlists(limit=PLAYLIST_LIMIT))
playlists

Wall time: 11.8 s


,title,playlistId,thumbnails,count
0,Your Likes,LM,[{'url': 'https://www.gstatic.com/youtube/medi...,NaN
1,'10s Electronic,RDCLAK5uy_lvnl-qLThmB2VUUwN33HvXwwCOesXrH1s,[{'url': 'https://lh3.googleusercontent.com/iN...,100
2,2020 Albums,PLWptjpDqazOzaFe2TUAPwytU_pyFMVBOh,[{'url': 'https://lh3.googleusercontent.com/Ms...,13
3,ADAS Spring,PLWptjpDqazOyeMKN81aSpU6UNEWZlbaiR,[{'url': 'https://lh3.googleusercontent.com/R2...,4
4,ambiant electro,PLWptjpDqazOzjaFOAlWV37AywCmro60ET,[{'url': 'https://lh3.googleusercontent.com/FD...,65
...,...,...,...,...
247,zzzz_all 6,PLWptjpDqazOyh08tZjD0SBzQbehN2PGrC,[{'url': 'https://lh3.googleusercontent.com/5r...,661
248,zzzz_thumbs_down,PLWptjpDqazOwUz2ci9ZEivIJ_4dNVx9v-,[{'url': 'https://lh3.googleusercontent.com/V6...,14
249,zzzz_thumbs_down 1,PLWptjpDqazOxj8cjCK6wF1KzhG1mT94vs,[{'url': 'https://lh3.googleusercontent.com/8U...,14
250,zzzz_thumbs_up,PLWptjpDqazOwbnb0x7ecV_RZWxJToctaF,[{'url': 'https://lh3.googleusercontent.com/U6...,547


In [6]:
%%time

PLAYLIST_SONG_LIMIT=10000
USER='Jake G'
REMOVE_DISLIKE = True
BACKUP_DIR  = './playlists/'
PLAYLIST_TSV_COLS = ['title', 'artist', 'album', 'likeStatus', 'duration', 'videoId', 'albumId', 'artistId']
METADATA_TSV = '_metadata.tsv'
METADATA_TSV_COLS = ['title','trackCount','duration','privacy','id']

metadata = []
for i, row in playlists.iterrows():   
    print('\n\n(%d/%d) Playlist: %s %s ' % (i+1, len(playlists), row['title'], 80*'*'))

    # Fetch playlist
    playlist_meta = yt.get_playlist(row['playlistId'], limit=PLAYLIST_SONG_LIMIT)
    playlist_meta.pop('thumbnails', None)
    tracks = playlist_meta.pop('tracks', None)
    metadata.append(playlist_meta)
    print(pd.DataFrame.from_dict(playlist_meta, orient='index'))
    if playlist_meta['trackCount'] == 0:
        print('Skipping: %s, due to zero tracks' % playlist_meta['title'])
        continue

    # Parse playlist tracks
    tracks = pd.DataFrame(tracks)
    if REMOVE_DISLIKE:
        try:
            tracks_disliked = tracks.loc[tracks['likeStatus'] == 'DISLIKE']
            if len(tracks_disliked) and playlist_meta['author']['name'] == USER:
                print('Removing %d tracks:\n%s' % (len(tracks_disliked), tracks_disliked['title']))
                yt.remove_playlist_items(playlist_meta['id'], tracks_disliked.to_dict('records'))  
                tracks = tracks.loc[tracks['likeStatus'] != 'DISLIKE']
        except Exception as e:
            print('\nFailed to remove dislikes...\n%s\n' % e) 

    tracks['artistId'] = tracks['artists'].dropna().apply(lambda x: x[0]['id']) # TODO handle > 1 artist
    tracks['artist'] = tracks['artists'].dropna().apply(lambda x: x[0]['name'])
    tracks['albumId'] = tracks['album'].dropna().apply(lambda x: x['id'])
    tracks['album'] = tracks['album'].dropna().apply(lambda x: x['name'])
    tracks = tracks[PLAYLIST_TSV_COLS]
    tracks.to_csv(os.path.join(BACKUP_DIR, '%s.tsv' % playlist_meta['title']), sep='\t', header=True)


metadata = pd.DataFrame(metadata)[METADATA_TSV_COLS]
metadata.to_csv(os.path.join(BACKUP_DIR, METADATA_TSV), sep='\t', header=True)



************************** (217/252) Playlist: zzz_seed_music_albums Part 12 ****************************************
                                                            0
id                         PLWptjpDqazOxucEO1dM_JN3xI0OsMpbP0
privacy                                               PRIVATE
title                           zzz_seed_music_albums Part 12
author      {'name': 'Jake G', 'id': 'UCDvJYHQoKPhpF-sGFUM...
duration                                             6+ hours
trackCount                                                344


**************************************** (218/252) Playlist: zzz_seed_music_albums Part 2 ****************************************
                                                            0
id                         PLWptjpDqazOzoAhOdU6OqL95mg1WLsMd8
privacy                                               PRIVATE
title                            zzz_seed_music_albums Part 2
author      {'name': 'Jake G', 'id': 'UCDvJYHQoKPhpF-sGFUM...
durat